In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, sum as _sum
from delta.tables import DeltaTable
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult
import os

print(os.environ["SPARK_VERSION"])
date_str = "2024-07-25"

In [0]:
# Define file paths based on date parameter

booking_data = f"/Volumes/incremental_load/default/for_incremental/booking_data/bookings_{date_str}.csv"
customer_data = f"/Volumes/incremental_load/default/for_incremental/customer_data/customers_{date_str}.csv"
print(booking_data)
print(customer_data)

In [0]:
# Reading booking data

booking_df = spark.read \
    .format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('quote', "\"") \
    .option('multiLine', 'true') \
    .load(booking_data)

In [0]:
booking_df.printSchema()
display(booking_df)

In [0]:
customer_df = spark.read \
    .format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('quote', "\"") \
    .option('multiLine', 'true') \
    .load(customer_data)

In [0]:
customer_df.printSchema()
display(customer_df)

In [0]:
# Data quality checks booking data

check_incremental = Check(spark, CheckLevel.Error, "Booking Data Check") \
    .hasSize(lambda x: x > 0) \
    .isUnique("booking_id", hint = "Booking id is not unique throught") \
    .isComplete("customer_id") \
    .isComplete("amount") \
    .isNonNegative("amount") \
    .isNonNegative("quantity") \
    .isNonNegative("discount") 

In [0]:
# Data quality checks customer data
check_scd = Check(spark, CheckLevel.Error, "Customer Data Check") \
    .hasSize(lambda x: x > 0) \
    .isUnique("customer_id") \
    .isComplete("customer_name") \
    .isComplete("customer_address") \
    .isComplete("email")
   

In [0]:
# Run verification suite on booking data
booking_dq_check = VerificationSuite(spark) \
    .onData(booking_df) \
    .addCheck(check_incremental) \
    .run()

In [0]:
# Run verification suite cutomer data
customer_dq_check = VerificationSuite(spark) \
    .onData(customer_df) \
    .addCheck(check_scd) \
    .run()

In [0]:
booking_dq_check_df = VerificationResult.checkResultsAsDataFrame(spark, booking_dq_check)
display(booking_dq_check_df)

In [0]:
customer_dq_check_df = VerificationResult.checkResultsAsDataFrame(spark, customer_dq_check)
display(customer_dq_check_df)

In [0]:
# check if verfication passed for booking data

if booking_dq_check.status != "Success":
    raise ValueError("Data quality checks failed for booking data.")



In [0]:
# check if verfication passed for booking data

if customer_dq_check.status != "Success":
    raise ValueError("Data Quality Checks Failed for Customer Data")


In [0]:
# Add ingestion timestamp to booking
booking_df_incremental = booking_df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# Joining booking data with customer data

df_joined = booking_df_incremental.join(customer_df, "customer_id")

In [0]:
# Business transaformation: calculate total amount after discount and filter

df_transformed = df_joined \
    .withColumn("total_cost", col("amount") - col("discount")) \
        .filter(col("quantity") > 0)

In [0]:
df_transformed_agg = df_transformed \
    .groupBy("booking_type", "customer_id") \
        .agg(
            _sum("total_cost").alias("total_amount_sum"),
            _sum("quantity").alias("total_quantity_sum")
        )

In [0]:
# Check if the delta table exists
fact_table_path = "incremental_load.default.booking_fact_table"
fact_table_exists = spark._jsparkSession.catalog().tableExists(fact_table_path)

In [0]:
if fact_table_exists:
    df_existing_fact = spark.read.format("delta").table(fact_table_path)

    df_combined = df_existing_fact.unionByName(df_transformed_agg, allowMissingColumns=True)
# perform another group by function on the combined data
    df_final_agg = df_combined \
        .groupBy("booking_type", "customer_id") \
        .agg(
            _sum("total_amount_sum").alias("total_amount_sum"),
            _sum("total_quantity_sum").alias("total_quantity_sum")
        )

else:
    # if fact table doesnot exists, use the aggregated transformed data directly
    df_final_agg = df_transformed_agg

display(df_final_agg)

#write the final aggregated data back to the Delta Table

df_final_agg.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(fact_table_path)




In [0]:
scd_table_path = "incremental_load.default.customer_dim"
scd_table_exists = spark._jsparkSession.catalog().tableExists(scd_table_path)

# check if the customer table exists

if scd_table_exists:
    # load the existing SCD table
    scd_table = DeltaTable.forName(spark, scd_table_path)
    display(scd_table.toDF())

    # perform SCD2 merge logic

    scd_table.alias("scd") \
        .merge(
            customer_df.alias("updates"),
            "scd.customer_id = updates.customer_id and scd.valid_to = '9999-12-31'"
        ) \
        .whenMatchedUpdate(set={
            "valid_to": "updates.valid_from", 
        }) \
        .execute()
    customer_df.write.format("delta").mode("append").saveAsTable(scd_table_path)
else:
    # if the SCD table does not exists, write the customer data as new Delta table
    customer_df.write.format("delta").mode("overwrite").saveAsTable(scd_table_path)